# Lecture 01a: ML Foundations — Optimizers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wusche1/Illiad_ML_Engineering/blob/main/lectures/01_a_ml_foundations/exercises/02_optimizers/notebook.ipynb)

In [ ]:
import os, importlib
if os.getenv('COLAB_RELEASE_TAG'):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/wusche1/Illiad_ML_Engineering/main/lectures/01_a_ml_foundations/exercises/02_optimizers/utils.py",
        "utils.py"
    )
    importlib.invalidate_caches()

import utils
importlib.reload(utils)
from utils import (
    rosenbrocks_banana, optimize, plot_banana,
    test_sgd, test_momentum, test_rmsprop, test_adam,
)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

## Rosenbrock's Banana Function

The job of an optimizer is to find the minimum of a loss function. Here we use Rosenbrock's banana function as a toy loss landscape:

$$f(x, y) = (a - x)^2 + b(y - x^2)^2 + 1$$

with $a=1, b=100$. The global minimum is at $(1, 1)$ where $f=1$. The narrow, curved valley makes this a challenging test for optimizers — you can clearly see how different algorithms navigate toward the minimum.

In [ ]:
def rosenbrocks_banana(x, y, a=1, b=100):
    return (a - x) ** 2 + b * (y - x**2) ** 2 + 1

trajectories = {}
plot_banana(trajectories)

## Optimization Helper

The `optimize` function from `utils.py` runs an optimizer for `n_steps` on the banana function, starting from a given parameter tensor, and records the trajectory. Each optimizer you build will be a class with `step()` and `zero_grad()` methods.

In [ ]:
def optimize(fn, parameters, optimizer, n_steps):
    trajectory = []
    for _ in range(n_steps):
        trajectory.append(parameters.detach().clone())
        loss = fn(*parameters)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    return torch.stack(trajectory).float()

---

## Exercise: Implement Optimizers from Scratch

You will implement four optimizers, each building on the last. For each one:
1. Fill in the `step()` method
2. Run the test cell to check correctness against PyTorch's implementation
3. Run the visualization cell to see the trajectory on the banana landscape

Trajectories accumulate across exercises so you can compare all optimizers side by side.

### Part A: Stochastic Gradient Descent

The simplest optimizer. Each step subtracts the gradient scaled by the learning rate:

$$\theta_{t+1} = \theta_t - \alpha \, \nabla_\theta \mathcal{L}$$

In [ ]:
class SGD:
    def __init__(self, parameters, lr=0.001):
        self.parameters = parameters
        self.lr = lr

    def step(self):
        with torch.no_grad():
            # TODO (1 line): update self.parameters using the gradient
            pass

    def zero_grad(self):
        self.parameters.grad.zero_()

In [ ]:
test_sgd(SGD)

<details>
<summary><b>Hint</b></summary>

Subtract the gradient, scaled by the learning rate, from `self.parameters`.
</details>

<details>
<summary><b>Solution</b></summary>

```python
class SGD:
    def __init__(self, parameters, lr=0.001):
        self.parameters = parameters
        self.lr = lr

    def step(self):
        with torch.no_grad():
            self.parameters -= self.lr * self.parameters.grad

    def zero_grad(self):
        self.parameters.grad.zero_()
```
</details>

In [ ]:
parameters = torch.tensor([-1.0, 2.0], requires_grad=True)
trajectories["SGD"] = optimize(rosenbrocks_banana, parameters, SGD(parameters, lr=0.001), 1000)
plot_banana(trajectories)

### Part B: SGD with Momentum

Momentum keeps a running velocity that smooths out noisy gradients and accelerates through flat regions. Think of a ball rolling downhill — it retains some of its previous velocity:

$$v_{t+1} = \beta \, v_t + (1 - \beta) \, \nabla_\theta \mathcal{L}$$
$$\theta_{t+1} = \theta_t - \alpha \, v_{t+1}$$

In [ ]:
class Momentum:
    def __init__(self, parameters, lr=0.001, momentum=0.9):
        self.parameters = parameters
        self.lr = lr
        self.momentum = momentum
        self.v = torch.zeros_like(parameters)

    def step(self):
        with torch.no_grad():
            # TODO (2 lines): update self.v, then update self.parameters
            pass

    def zero_grad(self):
        self.parameters.grad.zero_()

In [ ]:
test_momentum(Momentum)

<details>
<summary><b>Hint</b></summary>

Update the velocity as a weighted average of the old velocity and the current gradient. Then update the parameters using the velocity instead of the raw gradient.
</details>

<details>
<summary><b>Solution</b></summary>

```python
class Momentum:
    def __init__(self, parameters, lr=0.001, momentum=0.9):
        self.parameters = parameters
        self.lr = lr
        self.momentum = momentum
        self.v = torch.zeros_like(parameters)

    def step(self):
        with torch.no_grad():
            self.v = self.momentum * self.v + (1 - self.momentum) * self.parameters.grad
            self.parameters -= self.lr * self.v

    def zero_grad(self):
        self.parameters.grad.zero_()
```
</details>

In [ ]:
parameters = torch.tensor([-1.0, 2.0], requires_grad=True)
trajectories["Momentum"] = optimize(rosenbrocks_banana, parameters, Momentum(parameters, lr=0.01, momentum=0.9), 100)
plot_banana(trajectories)

### Part C: RMSProp

RMSProp adapts the learning rate per parameter by keeping a running average of the squared gradients:

$$r_{t+1} = \beta \, r_t + (1 - \beta) \, g_t^2$$
$$\theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{r_{t+1}} + \epsilon} \, g_t$$

Parameters with consistently large gradients get a smaller effective learning rate, while those with small gradients get a larger one.

In [ ]:
class RMSProp:
    def __init__(self, parameters, lr=0.001, beta=0.9, eps=1e-8):
        self.parameters = parameters
        self.lr = lr
        self.beta = beta
        self.eps = eps
        self.r = torch.zeros_like(parameters)

    def step(self):
        with torch.no_grad():
            # TODO (2 lines): update self.r, then update self.parameters
            pass

    def zero_grad(self):
        self.parameters.grad.zero_()

In [ ]:
test_rmsprop(RMSProp)

<details>
<summary><b>Hint</b></summary>

Update the running average of squared gradients. Then divide the gradient by the square root of that average (plus epsilon) when updating the parameters.
</details>

<details>
<summary><b>Solution</b></summary>

```python
class RMSProp:
    def __init__(self, parameters, lr=0.001, beta=0.9, eps=1e-8):
        self.parameters = parameters
        self.lr = lr
        self.beta = beta
        self.eps = eps
        self.r = torch.zeros_like(parameters)

    def step(self):
        with torch.no_grad():
            self.r = self.beta * self.r + (1 - self.beta) * self.parameters.grad ** 2
            self.parameters -= self.lr / (self.r.sqrt() + self.eps) * self.parameters.grad

    def zero_grad(self):
        self.parameters.grad.zero_()
```
</details>

In [ ]:
parameters = torch.tensor([-1.0, 2.0], requires_grad=True)
trajectories["RMSProp"] = optimize(rosenbrocks_banana, parameters, RMSProp(parameters, lr=0.2, beta=0.9, eps=1e-8), 100)
plot_banana(trajectories)

### Part D: Adam

Adam combines momentum and RMSProp. It keeps running averages of both the gradient (first moment $m$) and the squared gradient (second moment $v$), with bias correction to account for their zero initialization:

$$m_{t+1} = \beta_1 \, m_t + (1 - \beta_1) \, g_t$$
$$v_{t+1} = \beta_2 \, v_t + (1 - \beta_2) \, g_t^2$$
$$\hat{m} = \frac{m_{t+1}}{1 - \beta_1^{t+1}}, \quad \hat{v} = \frac{v_{t+1}}{1 - \beta_2^{t+1}}$$
$$\theta_{t+1} = \theta_t - \alpha \, \frac{\hat{m}}{\sqrt{\hat{v}} + \epsilon}$$

In [ ]:
class Adam:
    def __init__(self, parameters, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.parameters = parameters
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = torch.zeros_like(parameters)
        self.v = torch.zeros_like(parameters)
        self.t = 0

    def step(self):
        self.t += 1
        with torch.no_grad():
            g = self.parameters.grad
            # TODO: update self.m — biased first moment estimate
            # TODO: update self.v — biased second moment estimate
            # TODO: compute bias-corrected m_hat and v_hat
            # TODO: update self.parameters
            pass

    def zero_grad(self):
        self.parameters.grad.zero_()

In [ ]:
test_adam(Adam)

<details>
<summary><b>Hint</b></summary>

Update the first moment as an EMA of the gradient with weight $\beta_1$. Update the second moment as an EMA of the squared gradient with weight $\beta_2$. Divide each by $(1 - \beta^t)$ to correct for initialization bias. Update the parameters using the corrected first moment divided by the square root of the corrected second moment.
</details>

<details>
<summary><b>Solution</b></summary>

```python
class Adam:
    def __init__(self, parameters, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.parameters = parameters
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = torch.zeros_like(parameters)
        self.v = torch.zeros_like(parameters)
        self.t = 0

    def step(self):
        self.t += 1
        with torch.no_grad():
            g = self.parameters.grad
            self.m = self.beta1 * self.m + (1 - self.beta1) * g
            self.v = self.beta2 * self.v + (1 - self.beta2) * g ** 2
            m_hat = self.m / (1 - self.beta1 ** self.t)
            v_hat = self.v / (1 - self.beta2 ** self.t)
            self.parameters -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

    def zero_grad(self):
        self.parameters.grad.zero_()
```
</details>

In [ ]:
parameters = torch.tensor([-1.0, 2.0], requires_grad=True)
trajectories["Adam"] = optimize(rosenbrocks_banana, parameters, Adam(parameters, lr=0.9, beta1=0.9, beta2=0.999, eps=1e-8), 100)
plot_banana(trajectories)

---

### Bonus Challenge

Play around with the learning rates, betas, and epsilon.

**Challenge:** Get within $10^{-2}$ of the minimum $(1, 1)$ in as few steps as possible. Which optimizer and hyperparameters work best?

In [ ]:
# Your best attempt here!
parameters = torch.tensor([-1.0, 2.0], requires_grad=True)
# optimizer = ...
# traj = optimize(rosenbrocks_banana, parameters, optimizer, n_steps=???)
# print(f"Final position: {parameters.data}, distance to minimum: {(parameters.data - torch.tensor([1., 1.])).norm():.4f}")